In [1]:
!pip install --upgrade langchain langchain-core langchain-community langchain-google-genai duckduckgo-search ddgs wikipedia trafilatura requests pydantic langchainhub langchain-openai

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.9/136.9 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.6/134.6 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.5/120.5 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━

# Tools

In [2]:
import json, requests, trafilatura
from google.colab import userdata
from langchain_community.tools import tool
from pydantic import BaseModel

/tmp/ipykernel_2029/2664524995.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import tool


In [3]:
# web searching tool
@tool
def searchTool(query: str) -> dict:
  """Search Google for the given query and return the search results, including titles, URLs, snippets, and other metadata."""

  try:

    url = "https://google.serper.dev/search"

    payload = {
          "q": query
      }

    headers = {
      'X-API-KEY': userdata.get("SEARCH_API"),
      'Content-Type': 'application/json'
    }

    response = requests.request("POST", url, headers=headers, json=payload)

    if response.status_code == 200:
      search_results = response.json().get("organic", {})
      return ({
          'search_query': query,
          'search_results': search_results
        })

    else:
      return response.text

  except Exception as e:
    return {
        "tool name": "search Tool",
        "error": e
    }

print(searchTool.name)
print(searchTool.description)
print(searchTool.args)

searchTool
Search Google for the given query and return the search results, including titles, URLs, snippets, and other metadata.
{'query': {'title': 'Query', 'type': 'string'}}


In [4]:
# article scraper

@tool
def articleScraper(article_url: str) -> str:
    """Fetch a webpage and extract its main text content."""

    try:
        html = trafilatura.fetch_url(article_url)

        if not html:
            return "No HTML retrieved from the website. Try another URL."

        text = trafilatura.extract(html)

        if not text:
            return "Unable to extract text from the website HTML. Try another URL."

        return text

    except Exception as e:
        return f"Unknown error: {e}"

print(articleScraper.name)
print(articleScraper.description)
print(articleScraper.args)

articleScraper
Fetch a webpage and extract its main text content.
{'article_url': {'title': 'Article Url', 'type': 'string'}}


# Agent

In [5]:
from langchain.agents import create_agent

In [6]:
# prompt for React Agent
prompt = """
Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}
"""

In [7]:
# Model (LLm) For reasoning

from langchain_google_genai import ChatGoogleGenerativeAI

google_llm = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash",
    api_key = userdata.get("GEMINI_API_KEY")

)

from langchain_openai import ChatOpenAI

openai_llm = ChatOpenAI(
    model="deepseek/deepseek-v4-flash",
    base_url="https://api.aicredits.in/v1",
    api_key = userdata.get("OPENAI")

)

In [8]:

ReactAgent = create_agent(
    model=openai_llm,
    tools=[searchTool, articleScraper],
    system_prompt=prompt
)

In [9]:
response = ReactAgent.invoke({
    "messages":[
        {
            'role':'user',
            'content':'What are the top 10 trending TV shows and top 10 trending movies currently streaming on Netflix India in 2026? Rank them based on their popularity and viewing trends.'
        }
    ]
})

ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://flixpatrol.com/top10/netflix/india/
ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://www.reddit.com/r/netflixindia/comments/1r0pr5t/best_movie_or_series_of_2026_so_far/
ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://flixpatrol.com/top10/netflix/india/2026-07-09/
ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://flixpatrol.com/top10/streaming/india/2026-07-06/
ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://flixpatrol.com/top10/netflix/india/2026-07-08/


In [10]:
print(response['messages'][-1].content)

Now I have comprehensive data from multiple sources. Let me compile everything into a clear final answer!

---

# Top 10 Trending TV Shows & Movies on Netflix India (Current as of July 2026)

Based on data from **FlixPatrol**, **Netflix Tudum Official Top 10**, and **Zee News** rankings (week of June 29 – July 9, 2026).

---

## 📺 TOP 10 TRENDING TV SHOWS ON NETFLIX INDIA

| Rank | Show | Genre | Notes |
|:----:|------|-------|-------|
| **1** | **Lock Upp (Season 2)** | Reality / Social Experiment | Hosted by Riteish Deshmukh & Farah Khan. Currently the **#1 show** across all of India on FlixPatrol. |
| **2** | **Super Subbu (Season 1)** | Comedy / Drama | New Indian original starring Sundeep Kishan as a Sex Education Officer in a conservative village. Debuted July 2 and shot to #2 immediately. |
| **3** | **India's Got Latent (Season 2)** | Reality / Comedy | Samay Raina's wildly viral talent show. Spent 19+ consecutive days in the top 3. |
| **4** | **Avatar: The Last Airbender (Sea